# AURORA Ω MAX V3 — Champion Execution

This notebook is intentionally less theatrical and more professional.

V2 showed a small MAE improvement but failed ranking/IC and used immature 3Y targets. V3 fixes the evaluation contract first:

- no cached featured panel by default;
- rebuild features/lenses from the raw panel using current repo code;
- filter to common operating equities;
- mask 3Y returns whose horizon has not matured;
- train a deterministic train-only spine;
- run a conservative champion/challenger residual tournament;
- promote only if the challenger beats spine, uniform, and best single lens on mature validation with positive ranking diagnostics.

If no challenger wins, the product is still useful: AURORA Spine + Reverse DCF memo, with ML kept as shadow.


## 1. Runtime, Repo, Config


In [ ]:
import os, sys, json, time, random, subprocess, math
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
    import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, RandomForestRegressor

REPO_URL = "https://github.com/tbasaure-sys/fin.git"
REPO_REF = "main"
WORKDIR = Path("/content/fin") if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path("/content/drive/MyDrive/blsprime_aurora_omega") if IN_COLAB else Path("./_local_data/blsprime_aurora_omega")
PANEL_ROOT = DRIVE_ROOT / "panel"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"

START_YEAR = 2005
LAST_FEATURE_YEAR = 2024
DATA_CUTOFF_DATE = pd.Timestamp.utcnow().tz_localize(None).date().isoformat()
TRAIN_END_YEAR = 2020
CORE_END_YEAR = 2018
TUNE_START_YEAR = 2019
VAL_START_YEAR = 2021
HORIZON_YEARS = 3
TARGET = "ann_return_3y_fwd"
SEED = 7

for p in [DRIVE_ROOT, PANEL_ROOT, ARTIFACT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("Runtime:", {"python": sys.version.split()[0], "cuda": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
print("Drive:", DRIVE_ROOT)
print("Data cutoff:", DATA_CUTOFF_DATE)


## 2. Sync Repo and Imports


In [ ]:
if IN_COLAB:
    if not WORKDIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(WORKDIR)])
    subprocess.check_call(["git", "-C", str(WORKDIR), "fetch", "origin", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "checkout", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", REPO_REF])

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

import importlib
import scripts.run_aurora_router_local as router
importlib.reload(router)
from aurora_omega.data import LENS_NAMES

print("Repo:", WORKDIR)
print("Router split:", router.TRAIN_END_YEAR, router.VAL_START_YEAR)
print("Lenses:", LENS_NAMES)


## 3. Load Raw Panel, Rebuild Featured, Clean Universe


In [ ]:
PANEL_CANDIDATES = [
    PANEL_ROOT / "panel_autodiscover_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_cache_only_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_selfcontained_2005_2024_1500.parquet",
]

panel_path = next((p for p in PANEL_CANDIDATES if p.exists() and p.stat().st_size > 0), None)
if panel_path is None:
    raise FileNotFoundError("No raw panel found. Expected one of: " + ", ".join(map(str, PANEL_CANDIDATES)))

panel = pd.read_parquet(panel_path)
print("Loaded raw panel:", panel_path, panel.shape, "tickers:", panel["ticker"].nunique())

# Always rebuild featured from raw panel. Old featured parquet may contain stale schema or immature targets.
featured = router.add_features(panel.copy())
featured = router.add_lens_predictions(featured)
featured["omega_regime"] = featured.apply(router.classify_spine_regime, axis=1)
featured["omega_primary_question"] = featured["omega_regime"].map(router.primary_question_for_regime)
expectations = featured.apply(router.reverse_dcf_expectations, axis=1)
featured["omega_expectations_pressure"] = [e.get("valuation_pressure_score", np.nan) for e in expectations]
featured["omega_feasibility_score"] = [
    router.score_expectation_feasibility(row, e).get("score", np.nan)
    for (_, row), e in zip(featured.iterrows(), expectations)
]
featured["omega_downside_anchor_score"] = [
    router.anchor_lens_checks(row, router.classify_spine_regime(row), e).get("asset_value", {}).get("score", np.nan)
    for (_, row), e in zip(featured.iterrows(), expectations)
]

def common_equity_mask(frame):
    ticker = frame["ticker"].astype(str).str.upper()
    sector = frame.get("sector", pd.Series("Unknown", index=frame.index)).astype(str).str.lower()
    industry = frame.get("industry", pd.Series("Unknown", index=frame.index)).astype(str).str.lower()
    simple_symbol = ticker.str.match(r"^[A-Z]{1,5}$")
    not_fund_like = ~ticker.str.endswith("X")
    not_fin_product = ~industry.str.contains("fund|etf|trust|closed-end|index", na=False)
    not_blank_sector = ~sector.isin(["", "unknown", "nan", "none"])
    return simple_symbol & not_fund_like & not_fin_product & not_blank_sector

pre = featured.shape
removed_sample = sorted(set(featured.loc[~common_equity_mask(featured), "ticker"].astype(str)))[:30]
featured = featured.loc[common_equity_mask(featured)].sort_values(["ticker", "year"]).reset_index(drop=True)
print("Featured rebuilt:", pre, "->", featured.shape, "tickers:", featured["ticker"].nunique())
print("Removed sample:", removed_sample)

featured_path = PANEL_ROOT / "featured_panel_v3_rebuilt_common_equity.parquet"
featured.to_parquet(featured_path, index=False)
print("Saved rebuilt featured:", featured_path)
display(featured[["ticker","year","sector","industry","omega_regime","pred_reverseDcf","pred_assetValue",TARGET]].head())


## 4. Mature Target Contract


In [ ]:
def mask_immature_forward_returns(frame, target_col=TARGET, horizon_years=HORIZON_YEARS, cutoff_date=DATA_CUTOFF_DATE):
    out = frame.copy()
    asof = pd.to_datetime(out["asof_date"], errors="coerce")
    cutoff = pd.Timestamp(cutoff_date)
    if cutoff.tzinfo is not None:
        cutoff = cutoff.tz_localize(None)
    mature_date = asof + pd.DateOffset(years=horizon_years)
    matured = mature_date.notna() & (mature_date <= cutoff)
    out[f"{target_col}_matured"] = matured
    out.loc[~matured, target_col] = np.nan
    return out

data = mask_immature_forward_returns(featured)
for c in [TARGET, "year"]:
    data[c] = pd.to_numeric(data[c], errors="coerce")
data["year"] = data["year"].astype("Int64")

ACTIVE_3Y_LENSES = [name for name in LENS_NAMES if name != "capitalCycle" and f"pred_{name}" in data.columns]
lens_cols = [f"pred_{name}" for name in ACTIVE_3Y_LENSES]
for c in lens_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")

data = data.dropna(subset=["ticker", "year", TARGET] + lens_cols).copy()
data["year"] = data["year"].astype(int)

core_df = data[data["year"] <= CORE_END_YEAR].copy()
tune_df = data[data["year"].between(TUNE_START_YEAR, TRAIN_END_YEAR)].copy()
val_df = data[data["year"] >= VAL_START_YEAR].copy()

print("Active 3Y lenses:", ACTIVE_3Y_LENSES)
print("Rows:", {"core": len(core_df), "tune": len(tune_df), "val": len(val_df)})
print("Tickers:", {"core": core_df.ticker.nunique(), "tune": tune_df.ticker.nunique(), "val": val_df.ticker.nunique()})
print("Years:", {"core": (int(core_df.year.min()), int(core_df.year.max())), "tune": (int(tune_df.year.min()), int(tune_df.year.max())), "val": (int(val_df.year.min()), int(val_df.year.max())) if len(val_df) else None})
print("Validation rows by year:")
display(val_df.groupby("year").size().rename("rows").reset_index())

if len(core_df) < 1000 or len(tune_df) < 300 or len(val_df) < 300:
    raise RuntimeError("Not enough mature 3Y data after cleaning. Expand historical panel or lower universe filter intentionally.")


## 5. Train-Only Spine


In [ ]:
def mae_np(pred, y):
    pred = np.asarray(pred, dtype="float64")
    y = np.asarray(y, dtype="float64")
    ok = np.isfinite(pred) & np.isfinite(y)
    return float(np.mean(np.abs(pred[ok] - y[ok]))) if ok.any() else float("nan")

def ic_np(pred, y):
    s = pd.DataFrame({"pred": pred, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 20 or s["pred"].nunique() < 5 or s["y"].nunique() < 5:
        return float("nan")
    return float(s["pred"].rank().corr(s["y"].rank()))

def by_year_metrics(frame, pred_col, target_col=TARGET):
    rows = []
    for yr, sub in frame[["year", pred_col, target_col]].dropna().groupby("year"):
        if len(sub) < 20:
            continue
        rows.append({
            "year": int(yr),
            "rows": int(len(sub)),
            "mae": mae_np(sub[pred_col], sub[target_col]),
            "ic": ic_np(sub[pred_col], sub[target_col]),
        })
    return pd.DataFrame(rows)

def decile_spread(frame, pred_col, target_col=TARGET):
    spreads = []
    for _, sub in frame[["year", pred_col, target_col]].dropna().groupby("year"):
        if len(sub) < 80 or sub[pred_col].nunique() < 10:
            continue
        q = pd.qcut(sub[pred_col], 10, labels=False, duplicates="drop")
        if q.max() < 1:
            continue
        spreads.append(float(sub.loc[q == q.max(), target_col].mean() - sub.loc[q == q.min(), target_col].mean()))
    return float(np.mean(spreads)) if spreads else float("nan")

def prior_for_lenses(names):
    raw = []
    for n in names:
        if n == "reverseDcf": raw.append(0.36)
        elif n == "assetValue": raw.append(0.26)
        elif n == "residualIncome": raw.append(0.17)
        elif n == "roicFade": raw.append(0.08)
        elif n == "dcf": raw.append(0.07)
        elif n == "unitEconomics": raw.append(0.04)
        else: raw.append(0.02)
    raw = np.asarray(raw, dtype="float64")
    return raw / raw.sum()

def fit_simplex_spine(frame, lens_cols, target_col=TARGET, epochs=2500, lr=0.05, l2=0.04):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X = torch.tensor(frame[lens_cols].values.astype("float32"), device=device)
    y = torch.tensor(frame[target_col].values.astype("float32"), device=device)
    prior = torch.tensor(prior_for_lenses([c.replace("pred_", "") for c in lens_cols]).astype("float32"), device=device)
    theta = torch.zeros(len(lens_cols), device=device, requires_grad=True)
    opt = torch.optim.Adam([theta], lr=lr)
    for _ in range(epochs):
        w = torch.softmax(theta, dim=0)
        pred = X @ w
        loss = torch.nn.functional.smooth_l1_loss(pred, y, beta=0.04) + l2 * ((w - prior) ** 2).sum()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    with torch.no_grad():
        w = torch.softmax(theta, dim=0).cpu().numpy()
    return {col.replace("pred_", ""): float(weight) for col, weight in zip(lens_cols, w)}

spine_weights = fit_simplex_spine(core_df, lens_cols)

def apply_spine(frame, weights):
    pred = np.zeros(len(frame), dtype="float64")
    for name, weight in weights.items():
        pred += weight * frame[f"pred_{name}"].astype(float).values
    return pred

for frame in [core_df, tune_df, val_df]:
    frame["spine_pred"] = apply_spine(frame, spine_weights)
    frame["uniform_pred"] = frame[lens_cols].mean(axis=1).astype(float)

print("Spine weights:")
print(json.dumps(spine_weights, indent=2))
for label, frame in [("core", core_df), ("tune", tune_df), ("val", val_df)]:
    single = {c.replace("pred_", ""): mae_np(frame[c], frame[TARGET]) for c in lens_cols}
    print(label, {
        "spine_mae": mae_np(frame["spine_pred"], frame[TARGET]),
        "uniform_mae": mae_np(frame["uniform_pred"], frame[TARGET]),
        "best_single": min(single.items(), key=lambda kv: kv[1]),
        "spine_ic": ic_np(frame["spine_pred"], frame[TARGET]),
        "uniform_ic": ic_np(frame["uniform_pred"], frame[TARGET]),
        "spine_decile": decile_spread(frame, "spine_pred"),
    })
display(by_year_metrics(val_df, "spine_pred"))


## 6. Residual Tournament


In [ ]:
def make_model_frame(frame):
    forbidden = {TARGET, "year"}
    forbidden_prefixes = ("ann_return_", "price_t", "target_", "future_", "hard_", "omega_weight_", "pred_")
    numeric = []
    for c in frame.columns:
        if c in forbidden or c in lens_cols:
            continue
        if any(c.startswith(p) for p in forbidden_prefixes):
            continue
        if pd.api.types.is_numeric_dtype(frame[c]):
            numeric.append(c)
    cat = [c for c in ["omega_regime", "sector", "industry"] if c in frame.columns]
    return numeric, cat

feature_cols, cat_cols = make_model_frame(core_df)
print("Feature cols:", len(feature_cols), feature_cols[:30])
print("Cat cols:", cat_cols)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feature_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
    ],
    remainder="drop",
)

X_core_raw = core_df[feature_cols + cat_cols]
X_tune_raw = tune_df[feature_cols + cat_cols]
X_val_raw = val_df[feature_cols + cat_cols]
y_core = core_df[TARGET].astype(float).values
y_tune = tune_df[TARGET].astype(float).values
y_val = val_df[TARGET].astype(float).values
resid_core = y_core - core_df["spine_pred"].astype(float).values

challengers = {
    "ridge_residual": Ridge(alpha=8.0, random_state=SEED),
    "elastic_residual": ElasticNet(alpha=0.002, l1_ratio=0.10, random_state=SEED, max_iter=10000),
    "huber_residual": HuberRegressor(alpha=0.004, epsilon=1.35, max_iter=1000),
    "hgb_residual": HistGradientBoostingRegressor(
        loss="absolute_error",
        learning_rate=0.035,
        max_iter=240,
        max_leaf_nodes=18,
        l2_regularization=0.08,
        random_state=SEED,
    ),
    "extra_trees_residual": ExtraTreesRegressor(
        n_estimators=260,
        max_depth=7,
        min_samples_leaf=18,
        random_state=SEED,
        n_jobs=-1,
    ),
    "rf_residual": RandomForestRegressor(
        n_estimators=220,
        max_depth=8,
        min_samples_leaf=18,
        random_state=SEED,
        n_jobs=-1,
    ),
}

def tune_blend(anchor, residual_pred, y):
    best = None
    for rho in np.linspace(0.0, 0.75, 31):
        pred = anchor + rho * residual_pred
        mae = mae_np(pred, y)
        ic = ic_np(pred, y)
        score = mae - 0.004 * (0 if not np.isfinite(ic) else ic)
        row = {"rho": float(rho), "mae": mae, "ic": ic, "score": score}
        if best is None or row["score"] < best["score"]:
            best = row
    return best

results = []
fitted = {}
for name, model in challengers.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    pipe.fit(X_core_raw, resid_core)
    tune_resid = pipe.predict(X_tune_raw)
    val_resid = pipe.predict(X_val_raw)
    blend = tune_blend(tune_df["spine_pred"].astype(float).values, tune_resid, y_tune)
    tune_pred = tune_df["spine_pred"].astype(float).values + blend["rho"] * tune_resid
    val_pred = val_df["spine_pred"].astype(float).values + blend["rho"] * val_resid
    val_df[f"pred_{name}"] = val_pred
    tune_df[f"pred_{name}"] = tune_pred
    row = {
        "model": name,
        "rho": blend["rho"],
        "tune_mae": mae_np(tune_pred, y_tune),
        "tune_ic": ic_np(tune_pred, y_tune),
        "val_mae": mae_np(val_pred, y_val),
        "val_ic": ic_np(val_pred, y_val),
        "val_decile_spread": decile_spread(val_df, f"pred_{name}"),
    }
    results.append(row)
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values(["val_mae", "val_ic"], ascending=[True, False])
display(results_df)
winner_name = str(results_df.iloc[0]["model"])
winner_pred_col = f"pred_{winner_name}"
print("Winner:", winner_name)


## 7. Validation Scorecard


In [ ]:
lens_mae_val = {name: mae_np(val_df[f"pred_{name}"], val_df[TARGET]) for name in ACTIVE_3Y_LENSES}
best_single_name, best_single_mae = min(lens_mae_val.items(), key=lambda kv: kv[1])

scorecard = {
    "winner": winner_name,
    "val_rows": int(len(val_df)),
    "val_years": sorted([int(x) for x in val_df["year"].unique()]),
    "winner_mae": mae_np(val_df[winner_pred_col], val_df[TARGET]),
    "spine_mae": mae_np(val_df["spine_pred"], val_df[TARGET]),
    "uniform_mae": mae_np(val_df["uniform_pred"], val_df[TARGET]),
    "best_single_lens": best_single_name,
    "best_single_mae": best_single_mae,
    "winner_ic": ic_np(val_df[winner_pred_col], val_df[TARGET]),
    "spine_ic": ic_np(val_df["spine_pred"], val_df[TARGET]),
    "uniform_ic": ic_np(val_df["uniform_pred"], val_df[TARGET]),
    "winner_decile_spread": decile_spread(val_df, winner_pred_col),
    "spine_decile_spread": decile_spread(val_df, "spine_pred"),
    "lens_mae_val": lens_mae_val,
    "spine_weights": spine_weights,
}
print(json.dumps(scorecard, indent=2))

by_year = {}
for col in [winner_pred_col, "spine_pred", "uniform_pred"]:
    by_year[col] = by_year_metrics(val_df, col).to_dict(orient="records")
print("By-year metrics:")
print(json.dumps(by_year, indent=2))


## 8. Production Gate and Artifact Export


In [ ]:
def beats(a, b, margin=0.001):
    return np.isfinite(a) and np.isfinite(b) and a < b - margin

gates = {
    "mature_validation_rows": scorecard["val_rows"] >= 300,
    "beats_spine_mae": beats(scorecard["winner_mae"], scorecard["spine_mae"], 0.0015),
    "beats_uniform_mae": beats(scorecard["winner_mae"], scorecard["uniform_mae"], 0.0015),
    "beats_best_single_mae": beats(scorecard["winner_mae"], scorecard["best_single_mae"], 0.0010),
    "positive_winner_ic": np.isfinite(scorecard["winner_ic"]) and scorecard["winner_ic"] > 0.015,
    "ic_lift_vs_spine": np.isfinite(scorecard["winner_ic"]) and np.isfinite(scorecard["spine_ic"]) and scorecard["winner_ic"] > scorecard["spine_ic"] + 0.005,
    "positive_decile_spread": np.isfinite(scorecard["winner_decile_spread"]) and scorecard["winner_decile_spread"] > 0.015,
}
production_candidate = bool(all(gates.values()))

# If the residual challenger fails, the production policy explicitly uses the spine/memo layer.
product_mode = "residual_champion" if production_candidate else "spine_reverse_dcf_memo_with_ml_shadow"

out_dir = ARTIFACT_ROOT / f"omega_v3_champion_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}"
out_dir.mkdir(parents=True, exist_ok=True)

val_export = val_df[["ticker","year","asof_date",TARGET,"spine_pred","uniform_pred",winner_pred_col] + lens_cols].copy()
val_export.to_csv(out_dir / "validation_predictions.csv", index=False)
results_df.to_csv(out_dir / "challenger_tournament.csv", index=False)
(out_dir / "by_year_metrics.json").write_text(json.dumps(by_year, indent=2), encoding="utf-8")

manifest = {
    "version": "aurora_omega_max_v3_champion_execution",
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "artifact_dir": str(out_dir),
    "data_cutoff_date": DATA_CUTOFF_DATE,
    "production_candidate": production_candidate,
    "product_mode": product_mode,
    "gates": gates,
    "scorecard": scorecard,
    "challengers": results_df.to_dict(orient="records"),
    "decision": "PROMOTE_RESIDUAL_CHAMPION" if production_candidate else "DO_NOT_PROMOTE_ML_USE_SPINE_MEMO",
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2)[:12000])
print("Artifacts:", out_dir)


## 9. Investment Memo Layer


In [ ]:
def top_lenses_for_row(row, weights, k=3):
    vals = {}
    for name in ACTIVE_3Y_LENSES:
        vals[name] = float(row[f"pred_{name}"])
    by_abs = sorted(vals.items(), key=lambda kv: abs(kv[1]), reverse=True)[:k]
    by_weight = sorted(weights.items(), key=lambda kv: kv[1], reverse=True)[:k]
    return {
        "largest_signal_lenses": [{"lens": n, "prediction": v} for n, v in by_abs],
        "spine_weighted_lenses": [{"lens": n, "weight": float(w)} for n, w in by_weight],
    }

def primary_question(regime):
    return {
        "expensive_compounder": "Are market-implied expectations feasible?",
        "quality_compounder": "How long can excess ROIC persist?",
        "asset_heavy_cyclical": "Where are we in the supply response cycle?",
        "financial_book_capital": "Does book capital create value above cost of equity?",
        "commodity_resource": "Is normalized commodity economics better than spot expectations?",
        "pre_profit_platform": "Are unit economics improving enough to justify optionality?",
        "bottleneck_oligopoly": "Is scarcity durable and monetizable?",
        "regulated_utility_infrastructure": "Is regulated spread adequate versus cost of capital?",
    }.get(str(regime), "Which valuation question deserves trust first?")

memos = []
for _, row in val_df.sort_values(["ticker","year"]).groupby("ticker", as_index=False).tail(1).head(500).iterrows():
    regime = row.get("omega_regime", "general_intrinsic")
    pred = float(row[winner_pred_col]) if production_candidate else float(row["spine_pred"])
    memo = {
        "ticker": str(row["ticker"]),
        "year": int(row["year"]),
        "asof_date": str(row.get("asof_date", "")),
        "version": "aurora_v3_champion",
        "production_candidate": production_candidate,
        "product_mode": product_mode,
        "regime": str(regime),
        "primary_question": primary_question(regime),
        "selected_prediction_3y": pred,
        "spine_prediction_3y": float(row["spine_pred"]),
        "ml_shadow_prediction_3y": float(row[winner_pred_col]),
        "reverse_dcf_prediction_3y": float(row.get("pred_reverseDcf", np.nan)),
        "asset_value_prediction_3y": float(row.get("pred_assetValue", np.nan)),
        "lens_context": top_lenses_for_row(row, spine_weights),
        "falsifiers": [
            "Market-implied revenue growth is not visible in reported growth.",
            "Margin/ROIC deteriorates while the dominant lens assumes persistence.",
            "Capital cycle supply response weakens pricing power.",
            "Downside is not protected by asset value or book-capital spread.",
        ],
    }
    memos.append(memo)

mri_path = out_dir / "valuation_memos_v3.jsonl"
with open(mri_path, "w", encoding="utf-8") as f:
    for memo in memos:
        f.write(json.dumps(memo) + "\n")
print("Memo path:", mri_path)
display(pd.DataFrame(memos[:10]))
